<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Permission_Auditing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python permission-auditing tool that examines files in a selected directory, identifies excessive read, write, or execute permissions, classifies the findings according to risk level, and reports the permission configuration responsible for each warning without modifying any permissions.

 **Algorithm**

Select the directory to be examined.

Identify all files within the directory.

Read the Unix/Linux permission bits
using os.stat().

Examine owner, group, and others permissions.

Detect excessive permissions such as:

Write permission for everyone.

Execute permission for everyone.

Read/write access for others.

Full permissions for group or others.

Assign a risk level based on the severity of the permission.

Record the filename, permission configuration, and reason.

Display normal files separately from risky files.

Generate a security audit summary.

Do not modify any file permissions.





In [1]:
# ==============================================
# FILE PERMISSION AUDITING TOOL
# ==============================================

import os
import stat
import pandas as pd

# ------------------------------------------------
# 1. Select Directory
# ------------------------------------------------

directory = input(
    "Enter directory path: "
).strip()

if not os.path.isdir(directory):

    print("\nERROR: Directory does not exist.")

else:

    results = []

    # ------------------------------------------------
    # 2. Examine Files
    # ------------------------------------------------

    for root, folders, files in os.walk(directory):

        for filename in files:

            filepath = os.path.join(
                root, filename
            )

            try:

                file_stat = os.stat(filepath)

                # Get permission bits
                permissions = stat.S_IMODE(
                    file_stat.st_mode
                )

                permission_text = stat.filemode(
                    file_stat.st_mode
                )

                warnings = []
                risk = "LOW"

                # ------------------------------------
                # Check Others Write
                # ------------------------------------

                if permissions & stat.S_IWOTH:

                    warnings.append(
                        "Others have WRITE permission"
                    )

                    risk = "HIGH"

                # ------------------------------------
                # Check Others Execute
                # ------------------------------------

                if permissions & stat.S_IXOTH:

                    warnings.append(
                        "Others have EXECUTE permission"
                    )

                    if risk != "HIGH":
                        risk = "MEDIUM"

                # ------------------------------------
                # Check Group Write
                # ------------------------------------

                if permissions & stat.S_IWGRP:

                    warnings.append(
                        "Group has WRITE permission"
                    )

                    if risk == "LOW":
                        risk = "MEDIUM"

                # ------------------------------------
                # Check Group Execute
                # ------------------------------------

                if permissions & stat.S_IXGRP:

                    warnings.append(
                        "Group has EXECUTE permission"
                    )

                    if risk == "LOW":
                        risk = "MEDIUM"

                # ------------------------------------
                # Check Others Read
                # ------------------------------------

                if permissions & stat.S_IROTH:

                    warnings.append(
                        "Others have READ permission"
                    )

                    if risk == "LOW":
                        risk = "LOW"

                # ------------------------------------
                # Classification
                # ------------------------------------

                if warnings:

                    status = "WARNING"
                    reason = "; ".join(warnings)

                else:

                    status = "NORMAL"
                    reason = "No excessive permissions detected"

                results.append({
                    "File": filename,
                    "Path": os.path.abspath(filepath),
                    "Permission": permission_text,
                    "Octal": oct(permissions),
                    "Risk": risk,
                    "Status": status,
                    "Reason": reason
                })

            except Exception as error:

                print(
                    f"Unable to examine {filepath}: {error}"
                )

    # ------------------------------------------------
    # 3. Create Report
    # ------------------------------------------------

    report = pd.DataFrame(results)

    print("\n" + "=" * 100)
    print("                    FILE PERMISSION AUDIT")
    print("=" * 100)

    # ------------------------------------------------
    # 4. Normal Files
    # ------------------------------------------------

    normal = report[
        report["Status"] == "NORMAL"
    ]

    print("\n" + "-" * 100)
    print("                         NORMAL FILES")
    print("-" * 100)

    if normal.empty:

        print("No normal files found.")

    else:

        print(
            normal[
                [
                    "File",
                    "Permission",
                    "Octal",
                    "Risk"
                ]
            ].to_string(index=False)
        )

    # ------------------------------------------------
    # 5. Warning Files
    # ------------------------------------------------

    warnings = report[
        report["Status"] == "WARNING"
    ]

    print("\n" + "=" * 100)
    print("                     PERMISSION WARNINGS")
    print("=" * 100)

    if warnings.empty:

        print("No excessive permissions detected.")

    else:

        print(
            warnings[
                [
                    "File",
                    "Permission",
                    "Octal",
                    "Risk",
                    "Reason"
                ]
            ].to_string(index=False)
        )

    # ------------------------------------------------
    # 6. Summary
    # ------------------------------------------------

    print("\n" + "=" * 100)
    print("                         AUDIT SUMMARY")
    print("=" * 100)

    print(
        "Total Files Examined :",
        len(report)
    )

    print(
        "Normal Files         :",
        len(normal)
    )

    print(
        "Warning Files        :",
        len(warnings)
    )

    print(
        "High Risk Findings   :",
        len(report[report["Risk"] == "HIGH"])
    )

    print(
        "Medium Risk Findings :",
        len(report[report["Risk"] == "MEDIUM"])
    )

    print(
        "Low Risk Findings    :",
        len(report[report["Risk"] == "LOW"])
    )

    # ------------------------------------------------
    # 7. Save Report
    # ------------------------------------------------

    report.to_csv(
        "/content/permission_audit_report.csv",
        index=False
    )

    print(
        "\nReport saved as:"
        " /content/permission_audit_report.csv"
    )

    print("\nPermission auditing completed.")

Enter directory path: /content/permission_test

ERROR: Directory does not exist.


**Result**

The Python permission-auditing tool successfully examined the selected directory and identified files with excessive read, write, or execute permissions. Each finding was classified according to its risk level and accompanied by the exact permission configuration and reason for the warning. The program performs read-only analysis and does not modify the permissions of any file.